# Milestone 3: Train Your First Distributed Model 

This follows `data-preprocessing.ipynb` where we have the final dataframe saved to parquet to reduce resources and loading proccess.

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

In [2]:
PARQUET_PATH = "data/final_preprocessed"

df = spark.read.parquet(PARQUET_PATH)
df.show(5, truncate=80)

+------+--------+---+----+----+----+-------+------------------+----------+-------+-------------------------------+-------------------+--------------+-------------+-------------+----------------------+--------------------+
|  YEAR|STATEFIP|SEX| AGE|RACE|EDUC| INCTOT|        REALINCTOT| STATENAME|SEXNAME|                       RACENAME|           EDUCNAME|      STATE_OH|       SEX_OH|      RACE_OH|          REALINCTOT_Z|               AGE_Z|
+------+--------+---+----+----+----+-------+------------------+----------+-------+-------------------------------+-------------------+--------------+-------------+-------------+----------------------+--------------------+
|2024.0|     6.0|2.0|16.0| 6.0| 4.0|    0.0|               0.0|California| Female|Other Asian or Pacific Islander|           Grade 10|(51,[0],[1.0])|(2,[0],[1.0])|(9,[4],[1.0])| [-0.7464972771192954]| [-1.05138214502065]|
|2024.0|     6.0|1.0|67.0| 1.0| 7.0|27800.0|14761.800000000001|California|   Male|                          Whit

In [3]:
from pyspark.ml.feature import VectorAssembler

SEED = 42
sp = max(200, spark.sparkContext.defaultParallelism * 4)
spark.conf.set("spark.sql.shuffle.partitions", str(sp))

ml_df = df.select(F.col("EDUC").cast("double").alias("label"), "REALINCTOT_Z", "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH", "EDUCNAME")
assembler = VectorAssembler(inputCols=["REALINCTOT_Z", "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH"], outputCol="features")
ml_df = assembler.transform(ml_df)

train_df, val_df, test_df = ml_df.randomSplit([0.70, 0.15, 0.15], seed=SEED)

In [4]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf = RandomForestClassifier(labelCol="label", featuresCol="features", predictionCol="prediction", numTrees=20, maxDepth=10, seed=SEED)
model_baseline = rf.fit(train_df)

ev_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
ev_wp = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")

pred = model_baseline.transform(train_df)
print("train_baseline", ev_acc.evaluate(pred), ev_f1.evaluate(pred), ev_wp.evaluate(pred))
pred = model_baseline.transform(val_df)
print("val_baseline", ev_acc.evaluate(pred), ev_f1.evaluate(pred), ev_wp.evaluate(pred))
pred = model_baseline.transform(test_df)
print("test_baseline", ev_acc.evaluate(pred), ev_f1.evaluate(pred), ev_wp.evaluate(pred))

train_baseline 0.4464036913715675 0.3330495604933033 0.4149309718709399
val_baseline 0.4464724593776777 0.3330839927041556 0.415042073167692
test_baseline 0.44663472717629393 0.33324497704333866 0.41481988831464744


In [5]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf2 = RandomForestClassifier(labelCol="label", featuresCol="features", predictionCol="prediction", numTrees=30, maxDepth=12, seed=SEED)
model_rf_hp = rf2.fit(train_df)

In [6]:
ev_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
ev_wp = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")

pred = model_rf_hp.transform(train_df)
print("train_rf30_d12", ev_acc.evaluate(pred), ev_f1.evaluate(pred), ev_wp.evaluate(pred))
pred = model_rf_hp.transform(val_df)
print("val_rf30_d12", ev_acc.evaluate(pred), ev_f1.evaluate(pred), ev_wp.evaluate(pred))
pred = model_rf_hp.transform(test_df)
print("test_rf30_d12", ev_acc.evaluate(pred), ev_f1.evaluate(pred), ev_wp.evaluate(pred))

train_rf30_d12 0.4780522092075438 0.38988273382537103 0.4348835169596625
val_rf30_d12 0.47807499097469425 0.38984517507152205 0.4346803233300125
test_rf30_d12 0.4781798223623163 0.3899496459929954 0.435531688858845
